# Text Mining and Bias Detection Pipeline

This notebook demonstrates the complete text mining and bias detection pipeline for analyzing research projects and identifying discriminatory language and gender bias.

## Components:
1. **Text Preprocessing** - Tokenization, lemmatization, stopword removal
2. **Bias Detection** - Gender bias and discriminatory language identification
3. **Topic Modeling** - BERTopic for topic extraction
4. **Clustering** - Document grouping and similarity analysis
5. **Classification** - ML-based bias classification
6. **Decision Engine** - Personalized guidance and recommendations

## 1. Setup and Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Add project to path
project_path = os.path.abspath('.')
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from pipelines.text_mining_pipeline import TextMiningPipeline
from config.settings import PipelineConfig
from analysis.bias_detector import BiasDetector
from analysis.topic_modeler import BERTopicModeler
from models.clustering import DocumentClusterer
from models.classification import TextClassifier

print("✓ All imports successful")

## 2. Load and Prepare Sample Data

In [ ]:
# Create sample research project descriptions
sample_descriptions = [
    "We are looking for a strong male engineer to lead our team. The ideal candidate should be aggressive in pursuing market opportunities. We need a young leader with ambitious goals to drive our company forward.",
    
    "We are seeking a talented software engineer with strong problem-solving skills. The ideal candidate will have experience in team leadership and collaboration. We welcome applications from diverse backgrounds and perspectives.",
    
    "We need a beautiful, nurturing woman to handle customer relations. The position requires someone who is emotionally intelligent and care-oriented. Previous experience in supportive roles is preferred.",
    
    "We are recruiting for a technical project manager position. Required skills include project planning, resource management, and communication. We are committed to building a diverse and inclusive team.",
    
    "Join our research team as a data scientist. We value innovation and collaboration. We welcome candidates of all backgrounds to apply. Equal opportunities for career growth and development.",
    
    "Experienced male programmer needed for leadership position. Must be logical, aggressive in negotiations, and ambitious. Previous experience managing teams of 10+ people required.",
    
    "We are looking for young, energetic professionals (age 25-35) to join our startup. Must be quick learners with modern approaches. Avoid candidates with outdated skills.",
    
    "Position: Senior Analyst. We seek candidates with analytical expertise and attention to detail. We are an equal opportunity employer and value diversity in our team.",
]

# Create DataFrame
df = pd.DataFrame({
    'document_id': range(len(sample_descriptions)),
    'text': sample_descriptions
})

print(f"Loaded {len(df)} sample documents\n")
print(df.head(3))

## 3. Bias Detection Analysis

In [ ]:
# Initialize bias detector
bias_detector = BiasDetector()

# Analyze each document
bias_results = []
for idx, text in enumerate(df['text']):
    result = bias_detector.comprehensive_bias_analysis(text)
    result['document_id'] = idx
    bias_results.append(result)

# Create summary DataFrame
bias_summary = pd.DataFrame([
    {
        'document_id': r['document_id'],
        'bias_score': r['overall_bias_score'],
        'is_biased': r['is_biased'],
        'gender_bias_direction': r['gender_bias']['bias_direction'],
        'male_keywords': r['gender_bias']['male_keywords_found'],
        'female_keywords': r['gender_bias']['female_keywords_found']
    }
    for r in bias_results
])

print("Bias Detection Results:")
print(bias_summary)
print(f"\nDocuments with bias: {bias_summary['is_biased'].sum()} / {len(bias_summary)}")

## 4. Visualize Bias Distribution

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Bias scores
axes[0, 0].bar(bias_summary['document_id'], bias_summary['bias_score'], color='steelblue')
axes[0, 0].axhline(y=0.5, color='r', linestyle='--', label='Bias threshold')
axes[0, 0].set_xlabel('Document ID')
axes[0, 0].set_ylabel('Bias Score')
axes[0, 0].set_title('Overall Bias Scores')
axes[0, 0].legend()

# Plot 2: Biased vs Non-biased
bias_counts = bias_summary['is_biased'].value_counts()
axes[0, 1].pie(bias_counts.values, labels=['Non-biased', 'Biased'], autopct='%1.1f%%', colors=['green', 'red'])
axes[0, 1].set_title('Distribution of Biased Documents')

# Plot 3: Gender bias keywords
axes[1, 0].bar(bias_summary['document_id'], bias_summary['male_keywords'], label='Male keywords', alpha=0.7)
axes[1, 0].bar(bias_summary['document_id'], bias_summary['female_keywords'], label='Female keywords', alpha=0.7)
axes[1, 0].set_xlabel('Document ID')
axes[1, 0].set_ylabel('Keyword Count')
axes[1, 0].set_title('Gender-biased Keywords')
axes[1, 0].legend()

# Plot 4: Gender bias directions
gender_dist = bias_summary['gender_bias_direction'].value_counts()
axes[1, 1].barh(gender_dist.index, gender_dist.values, color=['lightcoral', 'lightyellow', 'lightblue'])
axes[1, 1].set_xlabel('Count')
axes[1, 1].set_title('Gender Bias Direction Distribution')

plt.tight_layout()
plt.show()

print("✓ Bias visualization complete")

## 5. Topic Modeling with BERTopic

In [ ]:
# Preprocess documents for topic modeling
from utils.preprocessing import TextPreprocessor
from config.settings import TextPreprocessingConfig

preprocessor = TextPreprocessor(TextPreprocessingConfig())
preprocessed_texts = preprocessor.preprocess_batch(df['text'].tolist())

print("Creating topic model...")
topic_modeler = BERTopicModeler(config=None)
topic_modeler.train(preprocessed_texts, verbose=False)

# Get topic information
topic_summary = topic_modeler.get_topic_summary()
topic_distribution = topic_modeler.get_topic_distribution()

print(f"\nTopics identified: {len(topic_summary)}")
print("\nTopic Summary:")
for topic_id, info in sorted(topic_summary.items()):
    print(f"Topic {topic_id}: {', '.join(info['keywords'][:3])}")

## 6. Document Clustering

In [ ]:
from config.settings import ClusteringConfig

# Configure and fit clustering
cluster_config = ClusteringConfig(n_clusters=3, algorithm='kmeans')
clusterer = DocumentClusterer(cluster_config)
clusterer.fit(df['text'].tolist())

# Get cluster distribution
cluster_dist = clusterer.get_cluster_distribution()
print("Cluster Distribution:")
for cluster_id, info in sorted(cluster_dist.items()):
    print(f"Cluster {cluster_id}: {info['count']} documents ({info['percentage']}%)")

# Get cluster evaluation
evaluation = clusterer.evaluate()
print(f"\nClustering Evaluation Metrics:")
for metric, value in evaluation.items():
    if metric != 'note':
        print(f"{metric}: {value:.4f}")

## 7. Full Pipeline Execution

In [ ]:
# Initialize and run full pipeline
config = PipelineConfig()
pipeline = TextMiningPipeline(config)

print("Running complete pipeline...\n")
results = pipeline.run_full_pipeline(df['text'].tolist())

# Get summary
summary = pipeline.get_summary()
print("\nPipeline Summary:")
for key, value in summary.items():
    print(f"{key}: {value}")

## 8. Generate Report

In [ ]:
# Generate comprehensive report
report = pipeline.generate_report()
print(report)

## 9. Detailed Bias Analysis for Specific Document

In [ ]:
# Analyze a specific document in detail
doc_index = 0  # Change this to analyze different documents
text = df['text'].iloc[doc_index]

print(f"\nDetailed Analysis of Document {doc_index}:")
print(f"Text: {text}\n")

analysis = bias_results[doc_index]
report = bias_detector.generate_bias_report(analysis)
print(report)

## 10. Export Results

In [ ]:
# Create output directory
output_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'output')
os.makedirs(output_dir, exist_ok=True)

# Export results
json_path = os.path.join(output_dir, 'analysis_results.json')
txt_path = os.path.join(output_dir, 'analysis_report.txt')

pipeline.export_results(json_path, format='json')
pipeline.export_results(txt_path, format='txt')

print(f"Results exported to {output_dir}/")